# 01 · Retrieval pipeline (SciFact smoke test, then TREC-COVID)

This notebook only *calls* the code in `src/`; all logic lives there. Run the cells top to bottom.

| Step | What happens | Output (where) |
|---|---|---|
| Setup | Mount Drive, clone/pull the repo, install Java + packages | `/content/robust-news-retrieval` |
| A3 | Documents → one clean JSONL file | Drive: `ir-project-data/<dataset>/corpus/docs.jsonl` |
| A4 | BM25: build Lucene index, search the queries | Drive: `<dataset>/bm25_index/`; repo: `runs/<dataset>/bm25_*.run` |
| A5 | Dense: encode all documents with E5 (GPU), then search | Drive: `<dataset>/dense/e5-base-v2/`; repo: `runs/<dataset>/dense_*.run` |
| A6 | RRF: fuse BM25 + dense rankings | repo: `runs/<dataset>/rrf_*.run` |
| A2 | Score the runs | repo: `results/<dataset>/*.csv` + a table printed below |
| Save | Copy runs and results to Drive so they survive the session | Drive: `ir-project-data/outputs/` |

**Use a GPU runtime** (Runtime → Change runtime type → T4 GPU): dense encoding needs it. Steps that are already done (corpus, BM25 index, finished encoding shards) are skipped automatically when you rerun.

## Setup
**First time in a session:** run the two cells below. The install cell then asks you to restart (**Runtime → Restart session**). After the restart, run both cells again: the install is skipped the second time.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/robust-news-retrieval'
BRANCH = 'main'

if not os.path.exists(REPO):
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q -b {BRANCH} https://{token}@github.com/HamsiniGururaj/robust-news-retrieval.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
import importlib.util

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'  # Pyserini needs Java 21
if importlib.util.find_spec('pyserini') is None:
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    !pip install -q -r requirements.txt
    print('Installed. Now do Runtime -> Restart session, then run the setup cells again.')
else:
    print('Packages already installed.')

In [ ]:
# Unit tests: should end with 'passed'
!python -m pytest -q

## Part 1 · SciFact smoke test
SciFact is small (5k documents), so this finishes in a couple of minutes and proves the pipeline works before the real dataset.

**What to expect:** the BEIR paper and Pyserini report BM25 nDCG@10 of roughly 0.66–0.68 on SciFact. Ours may be slightly lower because we deliberately turn stemming off. A value far below that (e.g. under 0.5) means something is wrong.

In [ ]:
!python -m src.prepare_corpus --dataset scifact
!python -m src.bm25 index --dataset scifact
!python -m src.bm25 search --dataset scifact
!python -m src.evaluate --dataset scifact --runs 'runs/scifact/*.run'

## Part 2 · TREC-COVID (primary benchmark)
171k documents. Corpus prep takes about a minute; indexing a few minutes.

The `index` step prints a check (`probe_document_frequencies`): **`the` must be > 0** (stopwords kept) and **`studi` must be 0** (no stemming). It stops with an error if not.

Two query sets are searched: `query` (the short keyword queries, our *original* queries) and `text` (the question form, secondary).

In [ ]:
!python -m src.prepare_corpus --dataset trec_covid
!python -m src.bm25 index --dataset trec_covid

In [ ]:
!python -m src.bm25 search --dataset trec_covid
!python -m src.bm25 search --dataset trec_covid --queries queries/trec_covid_text.tsv
!python -m src.evaluate --dataset trec_covid --runs 'runs/trec_covid/*.run'

## Part 3 · Dense retrieval (E5) and RRF
**Needs a GPU runtime.** `dense_encode` turns every document into a 768-number vector, saved to Drive in shards of 50,000 documents. If Colab disconnects, just rerun the cell: finished shards are skipped.

Rough time on a T4: SciFact well under a minute; TREC-COVID (171k short documents) in the order of 10 minutes.

`dense_search` then encodes each query (`query: ` prefix) and finds the closest documents with FAISS (exact search). `fuse` combines the BM25 and dense rankings with RRF (k=60).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

# SciFact: quick end-to-end check of dense + RRF
!python -m src.dense_encode --dataset scifact
!python -m src.dense_search --dataset scifact
!python -m src.fuse --dataset scifact --runs runs/scifact/bm25_text.run runs/scifact/dense_text.run
!python -m src.evaluate --dataset scifact --runs 'runs/scifact/*.run'

In [ ]:
# TREC-COVID: encode the corpus (resumable)
!python -m src.dense_encode --dataset trec_covid

In [ ]:
# TREC-COVID: dense search + RRF for both query sets, then score everything
!python -m src.bm25 search --dataset trec_covid
!python -m src.bm25 search --dataset trec_covid --queries queries/trec_covid_text.tsv
!python -m src.dense_search --dataset trec_covid
!python -m src.dense_search --dataset trec_covid --queries queries/trec_covid_text.tsv
!python -m src.fuse --dataset trec_covid --runs runs/trec_covid/bm25_query.run runs/trec_covid/dense_query.run
!python -m src.fuse --dataset trec_covid --runs runs/trec_covid/bm25_text.run runs/trec_covid/dense_text.run
!python -m src.evaluate --dataset trec_covid --runs 'runs/trec_covid/*.run'

**What to look for:** compare nDCG@10 across `bm25_*`, `dense_*` and `rrf_*`, and check **Judged@10** for the dense runs. If it is clearly lower than BM25's (~0.96), part of dense's top 10 was never judged, so its scores are underestimated (see `docs/PROGRESS.md` §2.5).

BM25 search is rerun above so its latency is measured the same way as dense (local index copy + warm-up query).

In [ ]:
# Efficiency numbers for the report (from the manifests)
import json, glob
for path in sorted(glob.glob('runs/trec_covid/*.run.manifest.json')):
    m = json.load(open(path))
    print(path.split('/')[-1].replace('.run.manifest.json', ''), '|', m.get('latency') or m.get('ms_per_query'),
          '|', m.get('index') or '', '|', m.get('encoding') or '')
print()
print('BM25 index:', json.load(open('/content/drive/MyDrive/ir-project-data/trec_covid/bm25_index.manifest.json'))['index_size_mb'], 'MB')

## Part 4 · Query rewrites (RQ2)
Run this **only after** all 100 rewrites are written, verified and committed (`queries/trec_covid_rewrite_a.tsv` and `_b.tsv` exist; see `docs/REWRITE_GUIDE.md`).

Needs the GPU runtime (dense query encoding). The corpus, BM25 index and document embeddings are reused from Drive, so nothing is re-encoded.

In [ ]:
for v in ['a', 'b']:
    q = f'queries/trec_covid_rewrite_{v}.tsv'
    !python -m src.bm25 search --dataset trec_covid --queries {q}
    !python -m src.dense_search --dataset trec_covid --queries {q}
    !python -m src.fuse --dataset trec_covid --runs runs/trec_covid/bm25_rewrite_{v}.run runs/trec_covid/dense_rewrite_{v}.run
!python -m src.evaluate --dataset trec_covid --runs 'runs/trec_covid/*.run'

In [ ]:
# Original vs rewrite, per system (Holm across the 3 systems in each family)
for v in ['a', 'b']:
    !python -m src.stats --dataset trec_covid --label rewrite_{v}_vs_original --pairs bm25_rewrite_{v}:bm25_query dense_rewrite_{v}:dense_query rrf_rewrite_{v}:rrf_query

## Save outputs to Drive
`runs/` and `results/` are inside the cloned repo, which is wiped when the session ends. This copies them to Drive (`ir-project-data/outputs/`). To commit them, download that folder and copy `runs/` and `results/` into your local repo.

In [ ]:
OUT = '/content/drive/MyDrive/ir-project-data/outputs'
!mkdir -p {OUT}
!cp -r runs results {OUT}/
!ls -R {OUT} | head -40